In [ ]:
!pip install -U scikit-learn gensim spacy nltk xgboost joblib matplotlib seaborn pandas tqdm
!python -m spacy download en_core_web_sm
!pip install contractions

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 91.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 61.1 MB/s eta 0:00:00
  Attempting uninstall: nltk
    Found existing installation: nltk 3.9.1
    Uninstalling nltk-3.9.1:
      Successfully uninstalled nltk-3.9.1
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: matplotlib
    Found existing insta

^C
^C


In [ ]:
!pip install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 14.8 MB/s eta 0:00:00


In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d kushagra3204/sentiment-and-emotion-analysis-dataset

Dataset URL: https://www.kaggle.com/datasets/kushagra3204/sentiment-and-emotion-analysis-dataset
License(s): apache-2.0
  0% 0.00/14.9M [00:00<?, ?B/s]
100% 14.9M/14.9M [00:00<00:00, 1.77GB/s]


In [ ]:
!unzip sentiment-and-emotion-analysis-dataset.zip

Archive:  sentiment-and-emotion-analysis-dataset.zip
  inflating: archive/combined_emotion.csv  
  inflating: archive/combined_sentiment_data.csv  


In [ ]:
import numpy as np
import pandas as pd
import os, re, string, json
from tqdm import tqdm
import joblib

import spacy
from gensim.models import Word2Vec
import nltk
from nltk.corpus import stopwords
import contractions

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tqdm.pandas()

In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('punkt_tab')

nlp = spacy.load("en_core_web_sm", disable=["parser","ner"])

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
df = pd.read_csv("/content/archive/combined_emotion.csv")
df.head()

,sentence,emotion
0,i just feel really helpless and heavy hearted,fear
1,ive enjoyed being able to slouch about relax a...,sad
2,i gave up my internship with the dmrg and am f...,fear
3,i dont know i feel so lost,sad
4,i am a kindergarten teacher and i am thoroughl...,fear


In [ ]:
df.shape

(422746, 2)

In [ ]:
df.isnull().sum()

,0
sentence,0
emotion,0


In [ ]:
df.duplicated().sum()

np.int64(6623)

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df['emotion'].value_counts()

,count
emotion,
joy,140779
sad,120989
anger,57235
fear,47664
love,34497
suprise,14959


We are doing needed text preprocessing here. we are removing stopwords but some of them like 'not', 'never' etc , we are not removing them because they contain meaning in the sentence .

In [ ]:
nltk_sw = set(stopwords.words('english'))
retain_tokens = {'no', 'not', 'never', 'none', 'nobody', 'nothing', 'neither', 'nor', 'very', 'too'}
stopwords_final = set([w for w in nltk_sw if w not in retain_tokens])

def preprocess_text(text, lemmatize=True, remove_stopwords=True):
  if not isinstance(text, str):
    return ""
  text = text.strip().lower()
  text = contractions.fix(text)

  text = re.sub(r'http\S+|www\.\S+', ' ', text)
  text = re.sub(r'\S+@\S+', ' ', text)

  # remove HTML tags
  text = re.sub(r'<.*?>', ' ', text)

  # remove punctuation
  text = re.sub(r'[%s]' % re.escape(string.punctuation), ' ', text)

  # remove extra whitespace
  text = re.sub(r'\s+', ' ', text).strip()

  # process with spaCy
  doc = nlp(text)
  tokens = []
  for token in doc:
    if token.is_space or token.is_punct or token.is_digit:
        continue
    lemma = token.lemma_.strip()
    if lemma == '':
        continue
    if remove_stopwords and lemma in stopwords_final:
        continue
    tokens.append(lemma)

  return " ".join(tokens), tokens

In [ ]:
# Example: create columns 'clean_text' and 'tokens'
cleaned = df['sentence'].progress_apply(lambda x: preprocess_text(x, lemmatize=True, remove_stopwords=True))
df['clean_text'] = cleaned.apply(lambda x: x[0])
df['tokens'] = cleaned.apply(lambda x: x[1])

100%|██████████| 416123/416123 [22:37<00:00, 306.44it/s]


In [ ]:
df[['sentence','clean_text']].head(10)

,sentence,clean_text
0,i just feel really helpless and heavy hearted,I feel really helpless heavy hearted
1,ive enjoyed being able to slouch about relax a...,I enjoy able slouch relax unwind frankly need ...
2,i gave up my internship with the dmrg and am f...,I give internship dmrg feel distraught
3,i dont know i feel so lost,I not know I feel lost
4,i am a kindergarten teacher and i am thoroughl...,I kindergarten teacher I thoroughly weary job ...
5,i was beginning to feel quite disheartened,I begin feel quite disheartened
6,i would think that whomever would be lucky eno...,I would think whomever would lucky enough stay...
7,i fear that they won t ever feel that deliciou...,I fear win ever feel delicious excitement chri...
8,im forever taking some time out to have a lie ...,I forever take time lie I feel weird
9,i can still lose the weight without feeling de...,I still lose weight without feel deprived


In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["emotion"])

Dividing data into train, test and validation data

In [ ]:
x = df['clean_text']
y = df['label']

x_train, x_test, y_train, y_test,idx_train, idx_test = train_test_split(x, y, df.index, test_size=0.2, random_state=42, stratify=y)
x_train_full, x_val, y_train_full, y_val = train_test_split(x_train, y_train, test_size=0.1, random_state=42, stratify=y_train)

Applying TF-IDF for feature extraction

In [ ]:
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,3),
    min_df=5,
    max_df=0.9,
    sublinear_tf=True
)
x_train_tfidf = tfidf.fit_transform(x_train_full)
x_val_tfidf = tfidf.transform(x_val)
x_test_tfidf = tfidf.transform(x_test)

In [ ]:
joblib.dump(tfidf, "tfidf_vectorizer.joblib")

['tfidf_vectorizer.joblib']

In [ ]:
def train_and_eval_model(model, X_tr, y_tr, X_v=None, y_v=None, X_te=None, y_te=None, name="model"):
  print(f"\n=== {name} ===")
  model.fit(X_tr, y_tr)
  if X_v is not None:
    y_pred_v = model.predict(X_v)
    print("Validation classification report:")
    print(classification_report(y_v, y_pred_v, digits=4))
    print("Val macro-F1:", f1_score(y_v, y_pred_v, average='macro'))
  if X_te is not None:
    y_pred = model.predict(X_te)
    print("Test classification report:")
    print(classification_report(y_te, y_pred, digits=4))
    print("Test macro-F1:", f1_score(y_te, y_pred, average='macro'))
    return model, y_pred

  return model, None

Training different models such as logistic regression, SVM, etc

In [ ]:
# Logistic Regression
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE)
lr, _ = train_and_eval_model(lr, x_train_tfidf, y_train_full, x_val_tfidf, y_val, x_test_tfidf, y_test, "LogisticRegression_TFIDF")
joblib.dump(lr, "lr_tfidf.joblib")


=== LogisticRegression_TFIDF ===
Validation classification report:
              precision    recall  f1-score   support

           0     0.8863    0.9275    0.9064      4579
           1     0.8616    0.8568    0.8592      3813
           2     0.9695    0.8876    0.9267     11262
           3     0.7271    0.9402    0.8200      2760
           4     0.9682    0.9195    0.9432      9679
           5     0.6665    0.9098    0.7693      1197

    accuracy                         0.9040     33290
   macro avg     0.8465    0.9069    0.8708     33290
weighted avg     0.9143    0.9040    0.9065     33290

Val macro-F1: 0.8708199026975095
Test classification report:
              precision    recall  f1-score   support

           0     0.8940    0.9251    0.9093     11447
           1     0.8640    0.8536    0.8587      9533
           2     0.9723    0.8905    0.9296     28156
           3     0.7249    0.9346    0.8165      6899
           4     0.9671    0.9270    0.9466     24198
   

['lr_tfidf.joblib']

In [ ]:
# LinearSVC
svc = LinearSVC(class_weight='balanced', max_iter=10000, random_state=RANDOM_STATE)
svc, _ = train_and_eval_model(svc, x_train_tfidf, y_train_full, x_val_tfidf, y_val, x_test_tfidf, y_test, "LinearSVC_TFIDF")
joblib.dump(svc, "svc_tfidf.joblib")


=== LinearSVC_TFIDF ===
Validation classification report:
              precision    recall  f1-score   support

           0     0.8933    0.9085    0.9008      4579
           1     0.8398    0.8458    0.8428      3813
           2     0.9395    0.9030    0.9209     11262
           3     0.7330    0.8264    0.7769      2760
           4     0.9529    0.9337    0.9432      9679
           5     0.6720    0.7703    0.7178      1197

    accuracy                         0.8950     33290
   macro avg     0.8384    0.8646    0.8504     33290
weighted avg     0.8989    0.8950    0.8964     33290

Val macro-F1: 0.8504002376034463
Test classification report:
              precision    recall  f1-score   support

           0     0.8968    0.9071    0.9019     11447
           1     0.8425    0.8397    0.8411      9533
           2     0.9400    0.9036    0.9215     28156
           3     0.7255    0.8175    0.7688      6899
           4     0.9534    0.9402    0.9468     24198
           5

['svc_tfidf.joblib']

In [ ]:
# MultinomialNB (works well with TF-IDF but sometimes without sublinear_tf)
mnb = MultinomialNB()
mnb, _ = train_and_eval_model(mnb, x_train_tfidf, y_train_full, x_val_tfidf, y_val, x_test_tfidf, y_test, "MultinomialNB_TFIDF")
joblib.dump(mnb, "mnb_tfidf.joblib")


=== MultinomialNB_TFIDF ===
Validation classification report:
              precision    recall  f1-score   support

           0     0.9325    0.8024    0.8625      4579
           1     0.8691    0.7747    0.8192      3813
           2     0.8053    0.9663    0.8785     11262
           3     0.9027    0.4841    0.6302      2760
           4     0.8661    0.9463    0.9044      9679
           5     0.9475    0.3016    0.4575      1197

    accuracy                         0.8521     33290
   macro avg     0.8872    0.7126    0.7587     33290
weighted avg     0.8609    0.8521    0.8413     33290

Val macro-F1: 0.7587254349931724
Test classification report:
              precision    recall  f1-score   support

           0     0.9304    0.8060    0.8637     11447
           1     0.8740    0.7748    0.8214      9533
           2     0.8076    0.9685    0.8807     28156
           3     0.9062    0.4804    0.6279      6899
           4     0.8705    0.9524    0.9096     24198
        

['mnb_tfidf.joblib']

In [ ]:
import joblib

tfidf_vectorizer = joblib.load("tfidf_vectorizer.joblib")
lr = joblib.load("lr_tfidf.joblib")
svc = joblib.load("svc_tfidf.joblib")
mnb = joblib.load("mnb_tfidf.joblib")

In [ ]:
joblib.dump(label_encoder, "label_encoder.joblib")

['label_encoder.joblib']

In [ ]:
def predict_emotion(sentence):
    sentence_tfidf = tfidf_vectorizer.transform([sentence])

    preds = {
        "LogisticRegression": label_encoder.inverse_transform(lr.predict(sentence_tfidf))[0],
        "LinearSVC": label_encoder.inverse_transform(svc.predict(sentence_tfidf))[0],
        "MultinomialNB": label_encoder.inverse_transform(mnb.predict(sentence_tfidf))[0],
    }

    return preds


In [ ]:
sentence = "i just feel really helpless and heavy hearted and afraid"
predictions = predict_emotion(sentence)

for model_name, emotion in predictions.items():
    print(f"{model_name} → {emotion}")

LogisticRegression → fear
LinearSVC → fear
MultinomialNB → fear


Here we can see here all three models works good to predict emotion from text

In [ ]:
tokenized_sentences = df['tokens'].tolist()
print("Example tokens:", tokenized_sentences[0])

Example tokens: ['I', 'feel', 'really', 'helpless', 'heavy', 'hearted']


Training word2vec for out dataset

In [ ]:
w2v = Word2Vec(
    sentences=tokenized_sentences,
    vector_size=300,
    window=5,
    min_count=3,
    workers=8,
    sg=1,
    epochs=15,
    compute_loss=True
)

for epoch in range(15):
    w2v.train(tokenized_sentences, total_examples=len(tokenized_sentences), epochs=1)
    print(f"Epoch {epoch+1} Loss: {w2v.get_latest_training_loss()}")


w2v.save("word2vec_model.model")
print("Word2Vec training completed.")


Epoch 1 Loss: 0.0


Epoch 2 Loss: 0.0


Epoch 3 Loss: 0.0


Epoch 4 Loss: 0.0


Epoch 5 Loss: 0.0


...

Epoch 10 Loss: 0.0


Epoch 11 Loss: 0.0


Epoch 12 Loss: 0.0


Epoch 13 Loss: 0.0


Epoch 14 Loss: 0.0
Epoch 15 Loss: 0.0
Word2Vec training completed.


In [ ]:
tokens_train_full = df.loc[x_train_full.index, 'tokens']
tokens_val = df.loc[x_val.index, 'tokens']
tokens_test = df.loc[x_test.index, 'tokens']

making average for all tokens in a single document using mean

In [ ]:
def avg_sentence_vector(tokens, model, vector_size=200):
  word_vectors = [model.wv[w] for w in tokens if w in model.wv]

  if len(word_vectors) == 0:
      return np.zeros(vector_size)

  return np.mean(word_vectors, axis=0)

Building average Vectors for all

In [ ]:
import numpy as np

def build_avg_vectors(token_series):
  matrix = np.zeros((len(token_series), w2v.vector_size))
  for i, tokens in enumerate(token_series):
      matrix[i] = avg_sentence_vector(tokens, w2v, w2v.vector_size)
  return matrix

X_train_avg = build_avg_vectors(tokens_train_full)
X_val_avg   = build_avg_vectors(tokens_val)
X_test_avg  = build_avg_vectors(tokens_test)


Applying standard scaler to standardize the data

In [ ]:
scaler_avg = StandardScaler()
X_train_avg = scaler_avg.fit_transform(X_train_avg)
X_val_avg   = scaler_avg.transform(X_val_avg)
X_test_avg  = scaler_avg.transform(X_test_avg)

joblib.dump(scaler_avg, "scaler_avg_w2v.joblib")

['scaler_avg_w2v.joblib']

In [ ]:
lr_avg, _ = train_and_eval_model(
    LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    X_train_avg, y_train_full, X_val_avg, y_val, X_test_avg, y_test,
    "LogisticRegression_Avg_W2V"
)

svc_avg, _ = train_and_eval_model(
    LinearSVC(class_weight='balanced', max_iter=10000, random_state=42),
    X_train_avg, y_train_full, X_val_avg, y_val, X_test_avg, y_test,
    "LinearSVC_Avg_W2V"
)


=== LogisticRegression_Avg_W2V ===
Validation classification report:
              precision    recall  f1-score   support

           0     0.6294    0.7109    0.6676      4579
           1     0.6227    0.7029    0.6603      3813
           2     0.8567    0.6773    0.7565     11262
           3     0.4482    0.7283    0.5549      2760
           4     0.8156    0.6664    0.7335      9679
           5     0.3858    0.8112    0.5229      1197

    accuracy                         0.6907     33290
   macro avg     0.6264    0.7161    0.6493     33290
weighted avg     0.7359    0.6907    0.7015     33290

Val macro-F1: 0.6492892223447252
Test classification report:
              precision    recall  f1-score   support

           0     0.6328    0.7197    0.6734     11447
           1     0.6336    0.7093    0.6693      9533
           2     0.8601    0.6820    0.7608     28156
           3     0.4527    0.7284    0.5583      6899
           4     0.8167    0.6706    0.7365     24198
 

In [ ]:
idf_dict = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))

making tfidf sentence vector for tfidf_word2vec

In [ ]:
def tfidf_sentence_vector(tokens, model, idf_dict, vector_size=200):
    vec = np.zeros(vector_size)
    weight_sum = 0.0

    for w in tokens:
        if w in model.wv and w in idf_dict:
            weight = idf_dict[w]
            vec += model.wv[w] * weight
            weight_sum += weight

    return vec / weight_sum if weight_sum != 0 else vec

In [ ]:
def build_tfidf_w2v_vectors(token_series):
    matrix = np.zeros((len(token_series), w2v.vector_size))
    for i, tokens in enumerate(token_series):
        matrix[i] = tfidf_sentence_vector(tokens, w2v, idf_dict, w2v.vector_size)
    return matrix

X_train_tfidf_w2v = build_tfidf_w2v_vectors(tokens_train_full)
X_val_tfidf_w2v   = build_tfidf_w2v_vectors(tokens_val)
X_test_tfidf_w2v  = build_tfidf_w2v_vectors(tokens_test)

In [ ]:
scaler_tfidf_w2v = StandardScaler()
X_train_tfidf_w2v = scaler_tfidf_w2v.fit_transform(X_train_tfidf_w2v)
X_val_tfidf_w2v   = scaler_tfidf_w2v.transform(X_val_tfidf_w2v)
X_test_tfidf_w2v  = scaler_tfidf_w2v.transform(X_test_tfidf_w2v)

joblib.dump(scaler_tfidf_w2v, "scaler_tfidf_w2v.joblib")

['scaler_tfidf_w2v.joblib']

In [ ]:
lr_tfidf_w2v, _ = train_and_eval_model(
    LogisticRegression(max_iter=1000, class_weight='balanced'),
    X_train_tfidf_w2v, y_train_full, X_val_tfidf_w2v, y_val, X_test_tfidf_w2v, y_test,
    "LogisticRegression_TFIDF_W2V"
)

svc_tfidf_w2v, _ = train_and_eval_model(
    LinearSVC(class_weight='balanced', max_iter=10000),
    X_train_tfidf_w2v, y_train_full, X_val_tfidf_w2v, y_val, X_test_tfidf_w2v, y_test,
    "LinearSVC_TFIDF_W2V"
)


=== LogisticRegression_TFIDF_W2V ===
Validation classification report:
              precision    recall  f1-score   support

           0     0.6088    0.6862    0.6452      4579
           1     0.6000    0.6808    0.6378      3813
           2     0.8424    0.6524    0.7353     11262
           3     0.4204    0.7087    0.5277      2760
           4     0.7994    0.6396    0.7106      9679
           5     0.3466    0.7769    0.4794      1197

    accuracy                         0.6657     33290
   macro avg     0.6029    0.6908    0.6227     33290
weighted avg     0.7172    0.6657    0.6782     33290

Val macro-F1: 0.6226782388430843
Test classification report:
              precision    recall  f1-score   support

           0     0.6155    0.6955    0.6530     11447
           1     0.6036    0.6841    0.6414      9533
           2     0.8440    0.6555    0.7379     28156
           3     0.4251    0.6978    0.5283      6899
           4     0.8010    0.6461    0.7153     24198

In [ ]:
from gensim.models import Word2Vec
w2v_model = Word2Vec.load("word2vec_model.model")
w2v = w2v_model.wv

In [ ]:
from collections import defaultdict

# Build vocabulary from Word2Vec
word_index = {word: i+1 for i, word in enumerate(w2v.key_to_index.keys())}
vocab_size = len(word_index) + 1
print("Vocabulary size:", vocab_size)


Vocabulary size: 23304


Trying DL models for same task

In [ ]:
def tokens_to_indices(token_list):
    return [word_index.get(token, 0) for token in token_list]  # 0 = OOV

df["token_indices"] = df["tokens"].apply(tokens_to_indices)

In [ ]:
X_train = [df.loc[i, "token_indices"] for i in x_train_full.index]
X_val   = [df.loc[i, "token_indices"] for i in x_val.index]
X_test  = [df.loc[i, "token_indices"] for i in x_test.index]

In [ ]:
from torch.nn.utils.rnn import pad_sequence
import torch

def pad_sequences(sequences, max_len=50):
    sequences = [torch.tensor(seq[:max_len]) for seq in sequences]  # trim long seqs
    return pad_sequence(sequences, batch_first=True, padding_value=0)

max_len = 50  # tune later
X_train_pad = pad_sequences(X_train, max_len)
X_val_pad   = pad_sequences(X_val, max_len)
X_test_pad  = pad_sequences(X_test, max_len)

y_train_tensor = torch.tensor(y_train_full.values)
y_val_tensor   = torch.tensor(y_val.values)
y_test_tensor  = torch.tensor(y_test.values)

In [ ]:
import numpy as np
embedding_dim = w2v.vector_size

embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, i in word_index.items():
    embedding_matrix[i] = w2v[word]

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

train_dataset = TensorDataset(X_train_pad, y_train_tensor)
val_dataset   = TensorDataset(X_val_pad, y_val_tensor)
test_dataset  = TensorDataset(X_test_pad, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=64)
test_loader  = DataLoader(test_dataset, batch_size=64)

In [ ]:
import torch.nn as nn

class LSTMEmotion(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(torch.tensor(embedding_matrix, dtype=torch.float32), freeze=False)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim*2, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        _, (h, _) = self.lstm(x)
        h = torch.cat((h[-2], h[-1]), dim=1)
        return self.fc(h)

model_lstm = LSTMEmotion(vocab_size, embedding_dim, 128, len(label_encoder.classes_))

In [ ]:
class GRUEmotion(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(torch.tensor(embedding_matrix, dtype=torch.float32), freeze=False)
        self.gru = nn.GRU(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim*2, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        _, h = self.gru(x)
        h = torch.cat((h[-2], h[-1]), dim=1)
        return self.fc(h)

model_gru = GRUEmotion(vocab_size, embedding_dim, 128, len(label_encoder.classes_))


In [ ]:
def train_model(model, train_loader, val_loader, epochs=5):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(epochs):
        model.train()
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            preds = model(X)
            loss = criterion(preds, y)
            loss.backward()
            optimizer.step()

        # Validation Accuracy
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                preds = model(X).argmax(1)
                correct += (preds == y).sum().item()
                total += y.size(0)

        print(f"Epoch {epoch+1}/{epochs} - Val Acc: {correct/total:.4f}")
    return model


In [ ]:
model_lstm = train_model(model_lstm, train_loader, val_loader, epochs=6)
torch.save(model_lstm.state_dict(), "lstm_word2vec.pt")

model_gru = train_model(model_gru, train_loader, val_loader, epochs=6)
torch.save(model_gru.state_dict(), "gru_word2vec.pt")

Epoch 1/6 - Val Acc: 0.9307
Epoch 2/6 - Val Acc: 0.9295
Epoch 3/6 - Val Acc: 0.9287
Epoch 4/6 - Val Acc: 0.9272
Epoch 5/6 - Val Acc: 0.9234
Epoch 6/6 - Val Acc: 0.9211
Epoch 1/6 - Val Acc: 0.9321
Epoch 2/6 - Val Acc: 0.9327
Epoch 3/6 - Val Acc: 0.9320
Epoch 4/6 - Val Acc: 0.9268
Epoch 5/6 - Val Acc: 0.9265
Epoch 6/6 - Val Acc: 0.9223


In [ ]:
def evaluate_test(model, test_loader, device):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test, y_test = x_test.to(device), y_test.to(device)
            outputs = model(x_test).argmax(1)
            all_preds.extend(outputs.cpu().numpy())
            all_labels.extend(y_test.cpu().numpy())

    print(classification_report(all_labels, all_preds, target_names=label_encoder.classes_))
    print("Test Macro-F1:", f1_score(all_labels, all_preds, average='macro'))

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
evaluate_test(model_lstm, test_loader, device)
evaluate_test(model_gru, test_loader, device)

              precision    recall  f1-score   support

       anger       0.92      0.95      0.93     11447
        fear       0.89      0.89      0.89      9533
         joy       0.93      0.95      0.94     28156
        love       0.81      0.77      0.79      6899
         sad       0.96      0.97      0.96     24198
     suprise       0.91      0.65      0.76      2992

    accuracy                           0.92     83225
   macro avg       0.90      0.86      0.88     83225
weighted avg       0.92      0.92      0.92     83225

Test Macro-F1: 0.8801960157918901
              precision    recall  f1-score   support

       anger       0.93      0.93      0.93     11447
        fear       0.86      0.90      0.88      9533
         joy       0.97      0.92      0.94     28156
        love       0.76      0.93      0.84      6899
         sad       0.96      0.96      0.96     24198
     suprise       0.79      0.72      0.75      2992

    accuracy                           0.92

In [ ]:
from torch.nn.utils.rnn import pad_sequence

def predict_sentence(model, sentence):
    model.eval()

    tokens = preprocess_text(sentence)[1]      # Get tokens
    token_ids = tokens_to_indices(tokens)      # Convert to indices
    seq = torch.tensor(token_ids[:max_len])    # Truncate
    seq = pad_sequence([seq], batch_first=True, padding_value=0)

    seq = seq.to(device)
    with torch.no_grad():
        pred = model(seq).argmax(1).item()

    return label_encoder.inverse_transform([pred])[0]

In [ ]:
predict_sentence(model_gru, "I am afraid for tomorrow's exam")

'fear'